# 01 — Exploración de Stanford GloVe (glove.6B.50d)

**Cita:** Pennington, J., Socher, R., & Manning, C. D. (2014). *GloVe: Global Vectors for Word Representation.* EMNLP 2014.

**Fuente:** https://nlp.stanford.edu/projects/glove/ (archivo `glove.6B.zip`, variante `glove.6B.50d.txt`)

Este cuaderno es el **plan de trabajo** para reemplazar los vectores sintéticos de `RAW_GLOVE_WORDS` (en `js/embeddings-data.js`) por vectores reales de GloVe. Las celdas de descarga están listas para ejecutarse una vez autorizada la descarga del dataset.

In [ ]:
import os
import io
import zipfile
import requests
import numpy as np
import pandas as pd

DATA_DIR = "../data/raw"
os.makedirs(DATA_DIR, exist_ok=True)
GLOVE_URL = "https://nlp.stanford.edu/data/glove.6B.zip"
GLOVE_ZIP_PATH = os.path.join(DATA_DIR, "glove.6B.zip")
GLOVE_TXT_PATH = os.path.join(DATA_DIR, "glove.6B.50d.txt")
print("Fuente oficial:", GLOVE_URL)

In [ ]:
# PASO 1 (se ejecuta en la siguiente fase, tras autorización de descarga)
# ADVERTENCIA: el archivo glove.6B.zip pesa ~822 MB.
#
# if not os.path.exists(GLOVE_TXT_PATH):
#     r = requests.get(GLOVE_URL, stream=True, timeout=120)
#     r.raise_for_status()
#     with open(GLOVE_ZIP_PATH, "wb") as f:
#         for chunk in r.iter_content(chunk_size=1 << 20):
#             f.write(chunk)
#     with zipfile.ZipFile(GLOVE_ZIP_PATH) as z:
#         z.extract("glove.6B.50d.txt", DATA_DIR)
print("Descarga pendiente de autorización explícita del usuario.")

In [ ]:
def load_glove(path, vocab=None):
    """Carga glove.6B.50d.txt en un dict {palabra: vector(50,)}.
    Si vocab no es None, filtra solo esas palabras (mas rapido).
    """
    embeddings = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            word = parts[0]
            if vocab is not None and word not in vocab:
                continue
            vec = np.asarray(parts[1:], dtype=np.float32)
            embeddings[word] = vec
    return embeddings

# Vocabulario equivalente al que hoy usa RAW_GLOVE_WORDS (version en ingles,
# ya que GloVe 6B esta entrenado sobre Wikipedia+Gigaword en ingles)
TARGET_VOCAB = {
    'king', 'queen', 'prince', 'princess', 'man', 'woman',
    'spain', 'madrid', 'france', 'paris', 'italy', 'rome',
    'germany', 'berlin', 'japan', 'tokyo',
    'computer', 'software', 'hardware', 'algorithm', 'intelligence',
    'robot', 'internet', 'code', 'server', 'data',
    'dog', 'cat', 'wolf', 'lion', 'tiger', 'elephant', 'eagle', 'whale',
    'joy', 'happiness', 'sadness', 'pain', 'love', 'anger', 'calm',
    'star', 'planet', 'galaxy', 'sun', 'atom', 'gravity', 'energy',
    'apple', 'bread', 'cheese', 'wine', 'coffee', 'chocolate'
}
print(f"Vocabulario objetivo: {len(TARGET_VOCAB)} palabras")

## Analogías reales (A - B + C = D)

Con los embeddings reales cargados, se recalcularán las analogías clásicas. A diferencia de los datos sintéticos (similitud ~0.99), con GloVe real se espera una similitud coseno **más modesta (0.70–0.90)**, que es la señal correcta de que el dato es auténtico.

In [ ]:
def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

def analogy(embeddings, a, b, c, topn=5):
    """Resuelve a - b + c = ? devolviendo las topn palabras mas cercanas."""
    target = embeddings[a] - embeddings[b] + embeddings[c]
    scores = []
    exclude = {a, b, c}
    for word, vec in embeddings.items():
        if word in exclude:
            continue
        scores.append((word, cosine_sim(target, vec)))
    scores.sort(key=lambda x: -x[1])
    return scores[:topn]

# Ejemplo de uso (tras cargar embeddings reales):
# emb = load_glove(GLOVE_TXT_PATH, vocab=TARGET_VOCAB)
# print(analogy(emb, 'king', 'man', 'woman'))
# print(analogy(emb, 'madrid', 'spain', 'france'))
print("Listo para ejecutar tras la descarga de datos reales.")

## Proyección 3D (PCA) con `js/projection-engine.js`

Una vez validados los vectores reales en Python, se exportarán a JSON (`word, category, vector[50]`) para alimentar `RAW_GLOVE_WORDS` en `js/embeddings-data.js`, reutilizando el motor PCA/t-SNE ya implementado en JavaScript sin modificarlo.

In [ ]:
# Ejemplo de exportación (tras cargar embeddings reales):
# records = [{"word": w, "vector": emb[w].tolist()} for w in TARGET_VOCAB if w in emb]
# pd.DataFrame(records).to_json("../data/processed/glove_words_real.json", orient="records", force_ascii=False)
print("Exportacion pendiente de la fase de descarga/integracion.")